In [ ]:
import sys
from pathlib import Path

REPO = Path("/content/snake_dqn")

if "google.colab" in sys.modules:
    if not REPO.exists():
        !git clone -q https://github.com/Mazako/snake_dqn.git {REPO}
    %cd {REPO}
    !git pull -q
    !pip install -q torchrl==0.14.1 tensordict==0.14.3

In [ ]:
from functools import partial

import torch
from torchrl.envs import Compose, RewardSum, SerialEnv, StepCounter, TransformedEnv

from snake_dqn.env import SnakeEnv

SIZE = 11
SEED = 0
N_ENVS = 8
DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

torch.manual_seed(SEED)
env = TransformedEnv(
    SerialEnv(N_ENVS, partial(SnakeEnv, SIZE, device=DEVICE)),
    Compose([StepCounter(), RewardSum()]),
)
env.set_seed(SEED)

In [ ]:
from tensordict.nn import TensorDictModule
from torch import nn
from torchrl.modules import (
    ActorValueOperator,
    OneHotCategorical,
    ProbabilisticActor,
    ValueOperator,
)

from snake_dqn.encoder import HIDDEN, make_trunk

common = TensorDictModule(
    make_trunk(SIZE), in_keys=["observation"], out_keys=["hidden"]
)
policy = ProbabilisticActor(
    TensorDictModule(nn.Linear(HIDDEN, 3), in_keys=["hidden"], out_keys=["logits"]),
    in_keys=["logits"],
    distribution_class=OneHotCategorical,
    spec=env.action_spec,
    return_log_prob=True,
)
value = ValueOperator(nn.Linear(HIDDEN, 1), in_keys=["hidden"])

actor_critic = ActorValueOperator(common, policy, value).to(DEVICE)
actor = actor_critic.get_policy_operator()
critic = actor_critic.get_value_operator()

In [ ]:
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

GAMMA = 0.99
LAMBDA = 0.95
CLIP_EPSILON = 0.2
ENTROPY_COEFF = 0.01
CRITIC_COEFF = 0.5

advantage_module = GAE(gamma=GAMMA, lmbda=LAMBDA, value_network=critic)
loss_module = ClipPPOLoss(
    actor,
    critic,
    clip_epsilon=CLIP_EPSILON,
    entropy_coeff=ENTROPY_COEFF,
    critic_coeff=CRITIC_COEFF,
    normalize_advantage=True,
)

In [ ]:
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement

FRAMES_PER_BATCH = 256 * N_ENVS
MINIBATCH_SIZE = 256

buffer = ReplayBuffer(
    storage=LazyTensorStorage(FRAMES_PER_BATCH, device=DEVICE),
    sampler=SamplerWithoutReplacement(),
    batch_size=MINIBATCH_SIZE,
)

In [ ]:
from statistics import mean

from torchrl.collectors import Collector

TOTAL_FRAMES = 1_000_000
N_EPOCHS = 4
LR = 2.5e-4
MAX_GRAD_NORM = 0.5
LOG_EVERY = 5

collector = Collector(
    env,
    actor,
    frames_per_batch=FRAMES_PER_BATCH,
    total_frames=TOTAL_FRAMES,
    auto_register_policy_transforms=True,
)
optimizer = torch.optim.Adam(loss_module.parameters(), lr=LR)

episode_rewards: list[float] = []
episode_scores: list[int] = []
episode_lengths: list[int] = []

for i, batch in enumerate(collector):
    done = batch["next", "done"].squeeze(-1)
    episode_rewards.extend(batch["next", "episode_reward"][done].squeeze(-1).tolist())
    episode_lengths.extend(batch["next", "step_count"][done].squeeze(-1).tolist())
    episode_scores.extend(batch["next", "score"][done].squeeze(-1).tolist())

    for _ in range(N_EPOCHS):
        with torch.no_grad():
            advantage_module(batch)
        buffer.extend(batch.reshape(-1))
        for minibatch in buffer:
            losses = loss_module(minibatch)
            loss = (
                losses["loss_objective"]
                + losses["loss_critic"]
                + losses["loss_entropy"]
            )
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(loss_module.parameters(), MAX_GRAD_NORM)
            optimizer.step()

    if i % LOG_EVERY == 0 and episode_rewards:
        print(
            f"frames={(i + 1) * FRAMES_PER_BATCH:>8} "
            f"episodes={len(episode_rewards):>6} "
            f"reward={mean(episode_rewards[-50:]):7.2f} "
            f"length={mean(episode_lengths[-50:]):6.1f} "
            f"score={mean(episode_scores[-50:]):5.2f} "
            f"objective={losses['loss_objective'].item():7.4f} "
            f"critic={losses['loss_critic'].item():7.4f} "
            f"entropy={losses['entropy'].item():.3f}"
        )

collector.shutdown()